# Learn to fly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/courses/gro860/labs/drone_ppo_learn_to_fly.ipynb)

A demo of using a reinforcement learning algorithm (PPO) to make a drone learn to fly.

**GRO860 exercise C.1.4 — Apprendre à voler.** Explore this example and answer:

1. How many training steps are needed to learn an adequate flight policy?
2. Does the learned policy generalize well? Test it with different initial conditions.
3. Observe the policy (control law) displayed at the end of training. How do you interpret it? Does it resemble a control law you know?
4. Modify the cost-function parameters to obtain a more "aggressive" flight behaviour (for example by penalizing energy use less). What do you observe?

We use the toolbox [minilink](https://github.com/alx87grd/minilink) for the dynamic equations, continuous-time simulation and animation of the drone, and [stable-baselines3](https://stable-baselines3.readthedocs.io) for the RL algorithm implementation.

In [ ]:
# Local: minilink already installed. Colab: clone + path + RL dependencies.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.costs import CostFunction
from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.aerial.drone import Drone2D
from minilink.interfaces.gymnasium import SB3Controller, Sys2Gym

# Dynamics

**Defining a dynamic system model**

Here we load an already defined class including all the dynamic equations of a planar drone with two thrusters, and we define the domain (for the states and inputs) for which we will generate a controller.

Note: to help the RL algorithm, we modify slightly the original class definition so that control inputs are normalized between -1 and 1, instead of values in Newtons of thrust. With the chosen scaling, $u = [0, 0]$ corresponds to a static gravity-compensation thrust, $u = [1, 1]$ to maximum thrust and $u = [-1, -1]$ to minimum thrust.

In [ ]:
class NormalizedDrone2D(Drone2D):
    """Planar drone with thrust inputs normalized between -1 and 1."""

    def __init__(self):
        super().__init__()

        # Parameters
        self.params["mass"] = 1.0  # kg
        self.params["inertia"] = 0.1  # kgm2

        # Normalized inputs
        self.inputs["u"].lower_bound = np.array([-1.0, -1.0])
        self.inputs["u"].upper_bound = np.array([+1.0, +1.0])
        self.inputs["u"].units = ["%", "%"]

        self.weight = self.params["gravity"] * self.params["mass"]
        self.thrust2weight = 1.2

        # Min/max states
        self.state.upper_bound = np.array([10, 10, 2 * np.pi, 10, 10, 10])
        self.state.lower_bound = -self.state.upper_bound

    def thrust(self, u):
        """Map a normalized input to thruster forces in Newtons."""
        return self.weight * ((self.thrust2weight - 1.0) * u + np.array([0.5, 0.5]))

    def f(self, x, u, t=0.0, params=None):
        return super().f(x, self.thrust(u), t, params)

    def get_dynamic_geometry(self, x, u, t=0, params=None):
        # Draw the thrust arrows using the de-normalized forces
        return super().get_dynamic_geometry(x, self.thrust(u), t, params)


plant = NormalizedDrone2D()

Quick open-loop flight demo with a small constant differential thrust:

In [ ]:
plant.x0 = np.array([-5.0, -2.0, 0.1, 0.0, 0.0, 0.1])

traj_open = plant.compute_forced(
    u=lambda t: np.array([0.01, -0.01]), tf=5.0, n_steps=2001, solver="euler"
)
plant.plot_trajectory(traj_open)
plant.animate(traj_open)

# Cost function

Here we define a cost function of the type

$$J = \int_{0}^{t_f} g(x, u, t) \, dt + h(x_f, t_f)$$

The reward maximized by the RL algorithm will be the negative of this cost, integrated over each time step: $r = -g(x,u,t)\,\Delta t$. The target state is hovering at the origin. **This is the cost to modify for question 4 of the exercise.**

In [ ]:
class CustomCostFunction(CostFunction):
    """
    J = int( g(x,u,t) * dt ) + h( x(T) , T )
    """

    def __init__(self):

        self.EPS = 0.1

        # Target state
        self.x_target = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

        # Quadratic cost weights
        self.Q = np.diag([1.0, 1.0, 6.0, 0.1, 0.1, 0.1])
        self.R = np.diag([0.001, 0.001])

        # Optional zone of zero cost if ||dx|| < EPS
        self.ontarget_check = False

    def g(self, x, u, t=0.0, params=None):
        """Quadratic additive running cost"""

        dx = x - self.x_target

        dJ = dx.T @ self.Q @ dx + u.T @ self.R @ u

        if self.ontarget_check:
            if np.linalg.norm(dx) < self.EPS:
                dJ = 0.0

        return dJ

    def h(self, x, t=0.0, params=None):
        """Terminal cost function with zero value"""

        return 0.0


cost = CustomCostFunction()

# Gym environment

We will use a standard "gym" environment. We have already defined the dynamics, the constraints and the cost function. The only additional thing to define is the distribution of initial states during the exploration.

In [ ]:
plant.x0 = np.zeros(6)  # nominal initial state: hover at the origin

env = Sys2Gym(plant, cost, dt=0.05)  # note the time step used for discrete time

# Gaussian distribution of initial states around x0
env.reset_mode = "gaussian"
env.x0_std = np.array([5.0, 5.0, 1.0, 1.0, 1.0, 0.2])

# Training

Headless PPO training. Start with a shorter run, look at the resulting policy and closed-loop behaviour below, then come back and train for more steps as needed (question 1 of the exercise).

In [ ]:
from stable_baselines3 import PPO

nn = PPO("MlpPolicy", env, verbose=1)

training_timesteps = 100000
nn.learn(training_timesteps)

More training as needed:

In [ ]:
# nn.learn(100000)

# Looking at the policy

Here we wrap the trained neural-network policy as a minilink feedback controller, and we plot a slice of the learned control law: the thrust commands as a function of the pitch angle $\theta$ and angular velocity $\omega$, with the other states at the target (question 3 of the exercise).

In [ ]:
ppo_ctl = SB3Controller(nn)


def plot_policy_slice(model, plant, ix, iy, anchor=None, n=60, axis=0):
    """Map of policy output u[axis] over the state plane (x[ix], x[iy])."""
    lb, ub = plant.state.lower_bound, plant.state.upper_bound
    xi = np.linspace(lb[ix], ub[ix], n)
    yi = np.linspace(lb[iy], ub[iy], n)
    anchor = np.zeros(plant.n) if anchor is None else np.asarray(anchor, dtype=float)

    obs = np.tile(anchor, (n * n, 1)).astype(np.float32)
    mesh_x, mesh_y = np.meshgrid(xi, yi)
    obs[:, ix] = mesh_x.ravel()
    obs[:, iy] = mesh_y.ravel()
    u, _ = model.predict(obs, deterministic=True)

    fig, ax = plt.subplots()
    mesh = ax.pcolormesh(
        xi, yi, u[:, axis].reshape(n, n), shading="gouraud", cmap="bwr"
    )
    mesh.set_clim(-1.0, 1.0)
    fig.colorbar(mesh, ax=ax)
    ax.set_xlabel(plant.state.labels[ix])
    ax.set_ylabel(plant.state.labels[iy])
    ax.set_title(f"PPO policy u[{axis}]")
    plt.show()


plot_policy_slice(nn, plant, ix=2, iy=5, axis=0)  # T1 vs (theta, omega)
plot_policy_slice(nn, plant, ix=2, iy=5, axis=1)  # T2 vs (theta, omega)

# Testing the closed-loop system

Here we simulate the drone in closed loop with the learned policy, starting away from the target (question 2: try other initial conditions here).

In [ ]:
plant.x0 = np.array([-5.0, -2.0, 1.0, 0.0, 0.0, 0.0])  # initial state

cl_sys = DiagramSystem()
cl_sys.add_subsystem(ppo_ctl, "ctl")
cl_sys.add_subsystem(plant, "plant")
cl_sys.connect("plant", "y", "ctl", "x")
cl_sys.connect("ctl", "u", "plant", "u")
cl_sys.name = "Drone with PPO controller"

traj = cl_sys.compute_trajectory(tf=5.0, n_steps=1001, solver="euler")
cl_sys.plot_trajectory(traj)

**Performance**

Here the performance, in terms of the defined cost function $J = \int g(x,u,t)\,dt$, is shown. Note that $\dot J = g(x,u,t)$ is the increment of cost at each instant and $J$ is the cumulative cost.

In [ ]:
# Rebuild the applied inputs from the policy, then evaluate the cost
u_sim, _ = nn.predict(traj.x.T.astype(np.float32), deterministic=True)

plant_traj = Trajectory(t=traj.t, x=traj.x, u=u_sim.T)
plant_traj = cost.evaluate_trajectory(plant_traj)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
axes[0].plot(plant_traj.t, plant_traj.signals["cost_rate"][0])
axes[0].set_ylabel("$\\dot{J} = g(x,u,t)$")
axes[0].grid(True, alpha=0.3)
axes[1].plot(plant_traj.t, plant_traj.signals["cost"][0])
axes[1].set_ylabel("$J = \\int g \\, dt$")
axes[1].set_xlabel("t [s]")
axes[1].grid(True, alpha=0.3)
plt.show()

print("Total trajectory cost J =", round(float(plant_traj.signals["cost"][0, -1]), 1))

**Animation of the simulation**

Here the following function generates an animation of the computed trajectory.

In [ ]:
cl_sys.animate(traj)